# Track B, step 3b: fine-tune the larger Granite model (149M)

**Question:** does the larger Granite embedding model, `granite-embedding-english-r2` (149M parameters, 768 numbers per text), do better than the small one we fine-tuned in step 3 (47M parameters, 384 numbers)?

**A fair test:** every setting is identical to step 3: the same training rows, the same 684 positives with 20 negatives each, positive weight 4, learning rate 3e-5, batch size 32, 3 epochs, and seed 0. Only the model changes, so any difference comes from model size (plus training luck; see the note at the end).

**Step 3's results to compare against, on 2025 validation:** small fine-tuned 5 hits@243, AUC 0.854. Baseline: 14.

The outputs get their own names (`*_large`), so step 3's results stay untouched.

**One memory change:** at 32 descriptions per batch, the 149M model ran out of the Mac's 24 GB. It now processes each batch of 32 as 4 chunks of 8, adds up their corrections, and makes one update. Mathematically this is the same as a batch of 32, so the training settings are unchanged.

## 1. Setup and settings

All settings are fixed here, before seeing any 2025 results, and follow the team plan. The 2025 validation set has only 52 positives, so we don't tune on it beyond a handful of runs.

In [ ]:
import os, re, time, math, random, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

DATA = next(p for p in [Path("../data"), Path("data")] if p.exists()).resolve()
OUT = DATA / "trackb"
OUT.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
SMOKE = os.environ.get("SMOKE") == "1"

CFG = dict(
    model="ibm-granite/granite-embedding-english-r2",
    max_length=256,        # covers 97% of descriptions in full (step 1)
    neg_per_pos=20,        # keep all positives + 20 random negatives each
    pos_weight=4.0,        # a missed positive costs 4x a false alarm in the loss
    micro_batch=8,         # 4 chunks of 8 per update = same as batch 32, less memory
    lr=3e-5, batch_size=32, epochs=3, warmup=0.06, weight_decay=0.01,
    holdout=0.10,          # share of training rows held back to watch the loss
    seed=0,
)
if SMOKE:
    CFG.update(epochs=1, neg_per_pos=5)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
seed_everything(CFG["seed"])
print("device:", DEVICE, "| smoke test:", SMOKE)
print(CFG)

## 2. Load the texts

Same rules as step 1: training is published 2018-2024 with labels by December 31, 2024; validation is the Phase 3 2025 set; scoring is the 2026 scoreboard. The input is `assigner | description`, with "exploited in the wild" wording removed.

The first run builds `data/trackb/texts.parquet` from the CSVs. Later runs, including on a GPU server, need only that one ~100 MB file.

In [ ]:
TEXTS = OUT / "texts.parquet"
IN_WILD = re.compile(r"exploited in the wild|actively exploited|may have been exploited|aware of (?:a )?report",
                     re.IGNORECASE)

def clean(description):
    text = re.sub(r",?\s*as (?:actively )?exploited in the wild[^.]*", "", description or "", flags=re.IGNORECASE)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return " ".join(s for s in sentences if not IN_WILD.search(s)).strip()

if not TEXTS.exists():
    import duckdb
    con = duckdb.connect()
    for t in ["vulnerabilities", "known_exploited", "scoreboard_2026"]:
        con.sql(f'''CREATE VIEW "{t}" AS
                    SELECT * FROM read_csv('{DATA}/{t}.csv', quote='"', escape='"', sample_size=-1)''')
    train = con.sql('''
        WITH v AS (SELECT *, CAST(published AS DATE) AS pub FROM vulnerabilities),
             k AS (SELECT cve_id, MIN(CAST(date_added AS DATE)) AS date_added FROM known_exploited GROUP BY cve_id)
        SELECT v.cve_id, v.assigner, v.description, year(v.pub) AS year, v.pub AS published,
               k.date_added AS kev_date,
               CAST(k.date_added IS NOT NULL AND k.date_added <= DATE '2024-12-31' AS INT) AS y
        FROM v LEFT JOIN k USING (cve_id)
        WHERE v.pub BETWEEN DATE '2018-01-01' AND DATE '2024-12-31'
          AND (k.date_added IS NULL OR k.date_added > last_day(v.pub))
        ORDER BY v.cve_id''').df().assign(split="train")
    val = (pd.read_parquet(DATA / "backtest/val_2025.parquet")
             [["cve_id", "assigner", "description", "y", "epss", "any_exploit", "published", "kev_date_added"]]
             .rename(columns={"kev_date_added": "kev_date"}).assign(split="val", year=2025))
    s26 = con.sql('''SELECT s.cve_id, v.assigner, v.description FROM scoreboard_2026 s
                     LEFT JOIN vulnerabilities v USING (cve_id) ORDER BY s.cve_id''').df().assign(split="score26", year=2026)
    texts = pd.concat([train, val, s26], ignore_index=True)
    texts["published"] = pd.to_datetime(texts["published"])
    texts["kev_date"] = pd.to_datetime(texts["kev_date"])  # kept so later steps can relabel with a later deadline
    texts["text"] = texts["assigner"].fillna("unknown") + " | " + texts["description"].fillna("").map(clean)
    texts.drop(columns=["description"]).to_parquet(TEXTS, index=False)
    print("built", TEXTS.name)

texts = pd.read_parquet(TEXTS)
texts["y"] = texts["y"].fillna(0).astype(int)  # 2026 rows have no label; 0 is a placeholder, never used
train = texts[texts.split == "train"].sort_values("cve_id", ignore_index=True)
val = texts[texts.split == "val"].sort_values("cve_id", ignore_index=True)
score26 = texts[texts.split == "score26"].sort_values("cve_id", ignore_index=True)
print(f"train {len(train):,} rows / {int(train.y.sum())} positives | val {len(val):,} / {int(val.y.sum())} | "
      f"2026 {len(score26):,}")
assert not texts["text"].str.contains(IN_WILD).any()

The Phase 3 scoring helpers: `hits_at_k`, `bootstrap` (90% range) and `compare` (paired test; a gain is real when its range excludes 0).

In [ ]:
BENCH_K, BENCH_N = 500, 72_670
default_k = lambda n: int(round(n * BENCH_K / BENCH_N))

def rank_order(df, score, seed=0):
    cols = [score] if isinstance(score, str) else score
    if isinstance(cols, list) and all(isinstance(c, str) for c in cols):
        keys = [df[c].to_numpy(dtype=float) for c in cols]
    else:
        keys = [np.asarray(score, dtype=float)]
    keys = [np.where(np.isnan(k), -np.inf, k) for k in keys]
    tie = np.random.default_rng(seed).permutation(len(df))
    return np.lexsort([tie] + [-k for k in reversed(keys)])

def hits_at_k(df, score, k=None, seed=0):
    k = k or default_k(len(df))
    return int(df["y"].to_numpy()[rank_order(df, score, seed)[:k]].sum())

def _ranks(df, score, seed=0):
    r = np.empty(len(df), dtype=np.int64)
    r[rank_order(df, score, seed)] = np.arange(len(df))
    return r

def _boot_hits(ranks, y, k, idx):
    return np.array([y[i][np.argpartition(ranks[i], k)[:k]].sum() for i in idx])

def bootstrap(df, score, k=None, n_boot=1000, ci=0.90, seed=0):
    k = k or default_k(len(df))
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    h = _boot_hits(_ranks(df, score, seed), df["y"].to_numpy(), k, idx)
    return int(np.quantile(h, (1 - ci) / 2)), int(np.quantile(h, 1 - (1 - ci) / 2))

def compare(df, score_a, score_b, k=None, n_boot=1000, ci=0.90, seed=0):
    k = k or default_k(len(df))
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    y = df["y"].to_numpy()
    d = (_boot_hits(_ranks(df, score_a, seed), y, k, idx)
         - _boot_hits(_ranks(df, score_b, seed), y, k, idx))
    return {"diff": hits_at_k(df, score_a, k, seed) - hits_at_k(df, score_b, k, seed),
            "range": (int(np.quantile(d, (1 - ci) / 2)), int(np.quantile(d, 1 - (1 - ci) / 2))),
            "p_a_better": round(float((d > 0).mean()), 3)}

BASELINE = ["any_exploit", "epss"]

## 3. The fine-tuning set: downsample the negatives

Training on all 165,885 rows would take hours, and 99.6% of them are negatives, so the model would mostly learn to say "no." Instead we keep **all 684 positives plus 20 random negatives for each** (about 14,000 rows). Then 10% is held out to watch the loss.

The **positive weight of 4** in the loss means missing an exploited bug costs four times as much as a false alarm. Only the ranking order matters, so the probabilities don't need to be calibrated.

In [ ]:
rng = np.random.default_rng(CFG["seed"])
pos_idx = np.flatnonzero(train.y.to_numpy() == 1)
neg_idx = np.flatnonzero(train.y.to_numpy() == 0)
if SMOKE:
    pos_idx = rng.choice(pos_idx, 64, replace=False)
neg_pick = rng.choice(neg_idx, CFG["neg_per_pos"] * len(pos_idx), replace=False)
ft = train.iloc[np.concatenate([pos_idx, neg_pick])].sample(frac=1, random_state=CFG["seed"]).reset_index(drop=True)
n_hold = int(len(ft) * CFG["holdout"])
hold, fit = ft.iloc[:n_hold].reset_index(drop=True), ft.iloc[n_hold:].reset_index(drop=True)
print(f"fine-tuning rows: {len(fit):,} ({int(fit.y.sum())} positives) | held out: {len(hold):,} ({int(hold.y.sum())} positives)")

## 4. The model and the training loop

`AutoModelForSequenceClassification` puts a new two-class layer on top of Granite. The loop below is plain PyTorch, so every step from the list at the top is visible:

- **AdamW** is the optimizer that applies the updates.
- The **learning rate** warms up over the first 6% of steps, then falls linearly to 0. Big early steps could wreck what Granite already knows.
- **Gradient clipping** caps any single update.
- On an NVIDIA GPU, **bfloat16** half-precision roughly doubles speed. On a Mac we stay in full precision.

In [ ]:
tok = AutoTokenizer.from_pretrained(CFG["model"])

def batches(df, batch_size, shuffle=False, seed=0):
    order = np.random.default_rng(seed).permutation(len(df)) if shuffle else np.arange(len(df))
    for i in range(0, len(order), batch_size):
        rows = df.iloc[order[i:i + batch_size]]
        enc = tok(rows["text"].tolist(), truncation=True, max_length=CFG["max_length"], padding=True, return_tensors="pt")
        yield {k: v.to(DEVICE) for k, v in enc.items()}, torch.tensor(rows["y"].fillna(0).to_numpy().astype(np.int64), device=DEVICE)

use_bf16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
autocast = (lambda: torch.autocast("cuda", dtype=torch.bfloat16)) if use_bf16 else (lambda: torch.autocast("cpu", enabled=False))

@torch.no_grad()
def predict(model, df, batch_size=32):
    # P(exploited) for every row. Sorting by length first means less padding, so it's faster.
    model.eval()
    order = df["text"].str.len().sort_values().index.to_numpy()
    out = np.empty(len(df), dtype=np.float32)
    sorted_df = df.loc[order]
    i = 0
    for enc, _ in batches(sorted_df.assign(y=0), batch_size):
        with autocast():
            logits = model(**enc).logits.float()
        p = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        out[i:i + len(p)] = p
        i += len(p)
    result = np.empty(len(df), dtype=np.float32)
    result[np.searchsorted(df.index.to_numpy(), order)] = out
    return result

def train_model(fit, hold, label="model"):
    seed_everything(CFG["seed"])
    model = AutoModelForSequenceClassification.from_pretrained(CFG["model"], num_labels=2).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    steps = CFG["epochs"] * math.ceil(len(fit) / CFG["batch_size"])
    sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * steps), steps)
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, CFG["pos_weight"]], device=DEVICE))
    log = []
    for epoch in range(CFG["epochs"]):
        model.train()
        t0, total, n = time.time(), 0.0, 0
        loss_sum = torch.nn.CrossEntropyLoss(weight=loss_fn.weight, reduction="sum")
        for step, (enc, y) in enumerate(batches(fit, CFG["batch_size"], shuffle=True, seed=CFG["seed"] + epoch)):
            batch_weight = loss_fn.weight[y].sum()   # normalizer of the full batch's weighted mean
            batch_loss = 0.0
            for i in range(0, len(y), CFG["micro_batch"]):
                part = {k: v[i:i + CFG["micro_batch"]] for k, v in enc.items()}
                with autocast():
                    loss = loss_sum(model(**part).logits.float(), y[i:i + CFG["micro_batch"]]) / batch_weight
                loss.backward()
                batch_loss += loss.item()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            if DEVICE == "mps" and step % 25 == 0:
                torch.mps.empty_cache()
            total += batch_loss * len(y); n += len(y)
        p_hold = predict(model, hold)
        p = np.clip(p_hold.astype(np.float64), 1e-7, 1 - 1e-7)  # weighted log-loss on the held-out rows, on the CPU
        w = np.where(hold.y.to_numpy() == 1, CFG["pos_weight"], 1.0)
        hold_loss = float(-(w * np.where(hold.y.to_numpy() == 1, np.log(p), np.log(1 - p))).sum() / w.sum())
        log.append({"epoch": epoch + 1, "train loss": round(total / n, 4), "held-out loss": round(hold_loss, 4),
                    "held-out AUC": round(float(roc_auc_score(hold.y, p_hold)), 3), "minutes": round((time.time() - t0) / 60, 1)})
        print(f"[{label}]", log[-1])
    return model, pd.DataFrame(log)

## 5. Train

In [ ]:
t_start = time.time()
model, train_log = train_model(fit, hold, "B3 large")
print(f"total training time: {(time.time() - t_start) / 60:.1f} min")
train_log

**How to read the training log:** the *training* loss should fall every epoch. The *held-out* loss and AUC are for rows the model never trained on, so they show whether it's learning patterns or memorizing. If held-out loss rises while training loss falls, the model is overfitting.

## 6. Score the 2025 validation set

This is the real test: 35,281 vulnerabilities the model has never seen, published after everything it trained on.

In [ ]:
eval_val = val.sample(3000, random_state=0).sort_values("cve_id", ignore_index=True) if SMOKE else val
t0 = time.time()
s_ft = predict(model, eval_val)
print(f"scored {len(eval_val):,} rows in {(time.time() - t0) / 60:.1f} min")

small_path = OUT / "text_val_2025_finetuned.csv"
small = (pd.read_csv(small_path).set_index("cve_id")["text_score"].reindex(eval_val.cve_id).to_numpy()
         if small_path.exists() else np.full(len(eval_val), np.nan))
frozen = pd.read_csv(OUT / "text_val_2025_frozen.csv").set_index("cve_id")["text_score"].reindex(eval_val.cve_id).to_numpy()
rows = []
for name, s in [("Baseline: exploit code then EPSS", BASELINE), ("Frozen Granite (step 2)", frozen),
                ("Fine-tuned small, 47M (step 3)", small), ("Fine-tuned large, 149M (this run)", s_ft)]:
    r = {"ranking": name, "hits@k": hits_at_k(eval_val, s), "90% range": "%d-%d" % bootstrap(eval_val, s)}
    if not isinstance(s, list):
        r["AUC"] = round(roc_auc_score(eval_val.y, s), 3)
    rows.append(r)
print("k =", default_k(len(eval_val)))
pd.DataFrame(rows).fillna("")

### Paired comparisons and the wider view

- **Fine-tuned vs. frozen:** did fine-tuning sharpen the text score?
- **Fine-tuned vs. baseline:** how does text alone compare with the best signals ranking?
- **Hits at larger cut-offs:** where in the ranking the gains come from.

In [ ]:
print("large vs. small:        ", compare(eval_val, s_ft, small))
print("large vs. frozen:       ", compare(eval_val, s_ft, frozen))
print("large vs. baseline:     ", compare(eval_val, s_ft, BASELINE))
pd.DataFrame([{"top k": k, "large": hits_at_k(eval_val, s_ft, k), "small": hits_at_k(eval_val, small, k), "frozen": hits_at_k(eval_val, frozen, k),
               "EPSS first": hits_at_k(eval_val, "epss", k), "exploit code then EPSS": hits_at_k(eval_val, BASELINE, k)}
              for k in [243, 500, 1000, 2000, 5000] if k < len(eval_val)])

### Blend with the signals baseline

The same rank average as step 2, with weights fixed in advance: 50/50 and 70/30 (baseline/text).

In [ ]:
rank_pct = lambda df, s: _ranks(df, s) / (len(df) - 1)  # 0 = best
rows = []
for w in [0.5, 0.7]:
    for name, s in [("frozen", frozen), ("fine-tuned", s_ft)]:
        b = -(w * rank_pct(eval_val, BASELINE) + (1 - w) * rank_pct(eval_val, s))
        c = compare(eval_val, b, BASELINE)
        rows.append({"blend (baseline/text)": f"{w:.0%} / {1 - w:.0%}", "text": name, "hits@k": hits_at_k(eval_val, b),
                     "vs. baseline": c["diff"], "diff range": "%+d to %+d" % c["range"], "P(better)": c["p_a_better"]})
pd.DataFrame(rows)

### What the description knew before EPSS did

These are exploited vulnerabilities that the fine-tuned model ranks highly while EPSS ranks them 5,000th or lower.

In [ ]:
ranked = eval_val.assign(text_rank=_ranks(eval_val, s_ft), epss_rank=_ranks(eval_val, "epss"))
pos = ranked[ranked.y == 1]
for top in [243, 2000]:
    print(f"positives in fine-tuned top {top:,} that EPSS ranks 5,000th or lower: "
          f"{((pos.text_rank < top) & (pos.epss_rank >= 5000)).sum()}")
pos.sort_values("text_rank")[["cve_id", "text_rank", "epss_rank", "text"]].head(15)

## 7. Save

- `text_val_2025_finetuned_large.csv`: the team handoff format (`cve_id, text_score`) for the Compare and combine table
- the fine-tuned model in `data/trackb/finetuned_large/`, for scoring 2026 and the 5-fold step (B4)

A smoke test saves nothing, so it never overwrites a real run.

In [ ]:
if SMOKE:
    print("Smoke test: nothing saved.")
else:
    pd.DataFrame({"cve_id": eval_val["cve_id"], "text_score": s_ft}).to_csv(OUT / "text_val_2025_finetuned_large.csv", index=False)
    model.save_pretrained(OUT / "finetuned_large"); tok.save_pretrained(OUT / "finetuned_large")
    train_log.to_csv(OUT / "finetune_log_large.csv", index=False)
    print("Saved:", *sorted(p.name for p in OUT.iterdir()))